In [ ]:
smart_sentences=["WHO-classificatie: 0 - zonder beperking in staat alle normale activiteiten uit te voeren",
"WHO-classificatie: 1 - beperkt in zware lich. activiteit maar ambulant en tot lichte arbeid in staat",
"WHO-classificatie: 2 - in staat voor zichzelf te zorgen, niet om te werken. >50% van de dag op de been",
"WHO-classificatie: 3 - slechts tot beperkte zelfverzorging in staat, minder dan 50% van de dag op de been",
"WHO-classificatie: 4 - volledig hulpbehoevend, gehele dag in bed of op stoel",
"Karnofsky-score: 100 - geen klachten, geen ziekteverschijnselen",
"Karnofsky-score: 90 - in staat tot normale activiteit; minimale tekenen van ziekte",
"Karnofsky-score: 80 - normale activiteit met enige moeite, enige symptomen van ziekte",
"Karnofsky-score: 70 - in staat tot zelfverzorging, niet tot werkzaamheden",
"Karnofsky-score: 50 - matig veel verzorging nodig, eveneens medische verzorging",
"Karnofsky-score: 40 - niet in staat tot persoonlijke verzorging of tot werken",
"Karnofsky-score: 30 - ernstig ziek, opname in ziekenhuis geindiceerd",
"KPS: 100% Geen klachten, geen ziekteverschijnselen.",
"KPS: 90% Minimale verschijnselen van de ziekte; in staat tot normale activiteit.",
"KPS: 80% Met inspanning tot normale activiteit in staat.",
"KPS: 70% In staat voor zichzelf te zorgen; onmogelijk om normale activiteiten te verrichten of om te werken",
"KPS: 60% Heeft af en toe hulp nodig, maar is in staat grotendeels voor zichzelf te zorgen",
"KPS: 50% Heeft veel hulp en frequente medische zorg nodig",
"KPS: 40%, grotendeels bedlegerig. Zeer pijnlijk",
"KPS: 30% Geheel bedlegerig; totale verzorging nodig; opname in ziekenhuis geïndiceerd"]
PS_scores=[0,1,2,3,4,0,1,1,2,3,3,4,0,1,1,2,2,3,3,4]

In [ ]:
def find_performance_scores(text):
    """
    Finds performance scores in a given text and extracts surrounding context.

    Args:
        text: The text to search.

    Returns:
        A list of dictionaries containing the score type, system (for ECOG/WHO),
        and the original surrounding context (if a match is found).
    """

    scores = []

    #Define patterns to capture WHO and ECOG scores, including all variations from PS_List
    patterns = [
        r"(?P<before>.{0,10})(WHO(?:[-\s]?score[:\s-]*)?(?:\d+[-\s]?\d*)|WHO(?:-schaal)?[:\s-]*\d+|WHO-classificatie[:\s-]*\d+|WHO-performance score[:\s-]*\d+|performance status[:\s-]*\d+|ECOG[:\s-]?\d+[-\s]?\d*)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(performance\s+status\s*(?:ECOG|WHO)\s*\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(Karnofsky[:\s-]*\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(KPS[:\s-]*\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(karnofsky-score[:\s-]*\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(Karnowsky performance status[:\s-]*\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(Karnofsky performance status[:\s-]*\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(PS(?:\s*\.\s*|\s*|\.|\s*PS\s*)\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(ECOG(?:\s*\.\s*|\s*|\.|\s*PS\s*)\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(WHO-performance score(?:\s*\.\s*|\s*|\.|\s*PS\s*)\d+)(?P<after>.{0,10})",
        r"(?P<before>.{0,10})(KPS(?:\s*\.\s*|\s*|\.|\s*PS\s*)\d+)(?P<after>.{0,10})"]


#(?P<before>.{0,10})
    for pattern in patterns:
        for match in re.finditer(pattern, text,re.IGNORECASE):
            
            score_type = "WHO" if "WHO" in match.group(2) else ("ECOG" if "ECOG" in match.group(2) else "Karnofsky" if "karnofsky-score" in match.group(2) else "Karnofsky" if "karnowsky" in match.group(2) else "Karnofsky" if "KPS" in match.group(2) else "Karnofsky" if "Karnofsky" in match.group(2) else "PS")
            
            score_system = match.group(2)
            context =match.group("before") +  match.group(2) + match.group("after") #match.group("before") + 
            #if score_type=='ECOG':
            #    print(context)
            scores.append({
                "type": score_type,
                "system": score_system,
                "context": context
            })

    return scores
def find_numeric_ps(text,pattern):
    scores=[]
    if type(text)==str:
        for match in re.finditer(pattern, text): #r"(\d+)(?!\s*(?:ml|kg|kcal))"
            score_value = match.group("number")
            #print(match.group(0))
            score_type = "WHO" if "WHO" in match.group(0) else ("ECOG" if "ECOG" in match.group(0) else "Karnofsky" if "Karnowsky" in match.group(0)  else "Karnofsky" if "karnofsky-score" in match.group(0) else "Karnofsky" if "KPS" in match.group(0) else "Karnofsky" if "Karnofsky" in match.group(0) else "PS")
            if score_type=='Karnofsky':
                score_value=score_value.replace('100','0').replace('90','1').replace('80','1').replace('70','2').replace('60','2').replace('50','3').replace('40','3').replace('30','4').replace('20','4')
            #else:
            #if (score_value == '100')  :
            #       converted_value = '0'
            #elif (score_value == '90') or (score_value == '80') :
            #       converted_value = '1'
            #elif (score_value == '70') or (score_value == '60') :
            #       converted_value = '2'
            #elif (score_value == '50') or (score_value == '40') :
            #      converted_value = '3'
            #elif (score_value == '30') or (score_value == '20') :
            #      converted_value = '4'
            #elif 
            #    converted_value=None
            if int(re.split(r"[\/,\-]",score_value)[0])<5:# in ['0','1','2','3','4','5','0.0','1.0','2.0','3.0','4.0','5.0']:  # Scores below 30 remain the same
                   converted_value = score_value
            else:
                converted_value=None
            #     converted_value = None
            
            # Karnofsky is here not converted, because it is used to censor the file.


            if converted_value is not None:
                scores.append(converted_value)
        return scores
def create_has_score_column(text):
    """
    Checks if there's a match in the text and returns the context if a match is found,
    otherwise returns an empty string.
    """
    scores = find_performance_scores(text)
    context=[score["context"] for score in scores]
    if scores:
        return (context) #scores[0]["context"])  # Assuming only the first match is needed scores[0]["context"]
    else:
        return ""
    
def extract_and_convert_scores(texts):
    """
    Extracts scores from a string, converts them based on ranges, and removes duplicates.

    Args:
    text: The text to extract scores from.

    Returns:
    A list of unique converted scores found in the text.
    """
    scores = []
    pattern = re.compile(r"""
\b
(?:                                      # --- allowed labels ---
    WHO\ PS|
    WHO-PS|
    WHO-score|
    WHO-schaal|
    WHO-classificatie|
    WHO\ classificatie|
    WHO|
    WHO-|
    ECOG|
    karnofsky|
    karnofsky-score|
    Karnowsky\ performance\ status| #Added
    Karnofsky\ performance\ status|
    KPS|
    WHO-performance\ score| 
    performance\ status|
    PS
)
\s*(?:[:=]?\s*)?                         # optional separator

(?P<number>                              # --- capture numbers here ---
    (?:100|[1-9]0|[0-4])                 # 0–5 or 10–100
    #(?:\s*[-]\s*(?:[0-4]))?            # optionally: 3-4, 2/3, 1,2
    (?:\s*[\/\-]\s*(?:[0-4]|[1-9]0|100))?            # optionally: 3-4, 2/3, 1,2
    #((?:[0-5]|[1-4]0|100)\s*[\/,\-]\s*(?:[0-5]|[1-9]0|100))?
)
\b
(?!
    \s*                                  # optional space
    (?:
        ml(?:/u(?:ur)?)? |               # ml, ml/u, ml/uur
        kcal  |  kg
    )
    \b                                   # enforce word boundary
)
""", re.IGNORECASE | re.VERBOSE)
    
        
    for text in texts:
        
        scores= find_numeric_ps(text,pattern)
        if len(scores)>0:
            break
            
    if len(scores)>1:
        print(scores)
        print(text)
        if scores[0]==scores[1]:
            scores=scores[0]
        else:
            scores=scores[0] + '-' + scores[1]
    elif len(scores)==1:
        scores=scores[0]
    elif len(scores)==0:
        scores='-1'
    return scores #list(set(scores))  # Ensure unique elements
#else:
#    print(text)
#    return str(text)


        
def extract_PS(df,col_text):
    # Add a new column named 'has_score' to store the context if a match is found
    #data['has_score'] = data['abstract'].apply(create_has_score_column)
    df['has_score'] = df[col_text].apply(create_has_score_column)
    #df['score type'] = df[col_text].apply(create_score_type_column)


    #data['censored']=data['abstract']


    #data.pseudonomized_text=data.abstract
    for i in range(len(smart_sentences)):
        sent=smart_sentences[i]
        ps=PS_scores[i]
        for j in df[df[col_text].isna()==False].index:
            if sent in df[col_text].loc[j]:
                df.loc[j,'PS']=ps
        
    # Create a new column named 'PS' to store the extracted and converted scores
    df['PS'] = df['has_score'].apply(extract_and_convert_scores)
    

    df['PS'].replace('0.0','0',inplace=True)
    df['PS'].replace('1.0','1',inplace=True)
    df['PS'].replace('2.0','2',inplace=True)
    df['PS'].replace('3.0','3',inplace=True)
    df['PS'].replace('4.0','4',inplace=True)
    
    pss=[]
    for i in range(len(df)):
        ps=df['PS'].iloc[i].replace(' ','')
       
        print(ps)
        if len(re.split(r"[\/,\-]",ps))>1:
            print(ps)
            try:
                int(re.split(r"[\/,\-]",ps)[0]) 
            except:
                ps=ps
            else:
                if int(re.split(r"[\/,\-]",ps)[0])>int(re.split(r"[\/,\-]",ps)[1]):
                    ps=re.split(r"[\/,\-]",ps)[1] + '-' + (re.split(r"[\/,\-]",ps)[0])
                elif int(re.split(r"[\/,\-]",ps)[0])==int(re.split(r"[\/,\-]",ps)[1]):
                    ps=re.split(r"[\/,\-]",ps)[0]
                elif int(re.split(r"[\/,\-]",ps)[0])<int(re.split(r"[\/,\-]",ps)[1]):
                    ps=re.split(r"[\/,\-]",ps)[0] + '-' + (re.split(r"[\/,\-]",ps)[1])
        pss.append(ps)
    df['PS']=pss

    return df

    

In [ ]:
df_notes=pd.read_csv('progress_notes.csv')
df_notes=df_notes[df_notes.pseudonomised_text.isna()==False].reset_index()
df=extract_PS(df_notes,"pseudonomised_text") #"pseudonomised_text")
df.to_csv(r'Z:\PS_classification\notes_classified_regex.csv')

-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
2
0
-1
-1
0
0
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
2
-1
-1
-1
-1
2
2
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
2
0
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
0
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
2
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1

-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
2
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
0
-1
-1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
0
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1


-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
0
0
-1
-1
0
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
0
-1
-1
-1
-1
2
2
0
0
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
0
-1
-1
-1
-1
-1
-1
0
0
0
-1
-1
1
0
2
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
0
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
0
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
1
-1
-1
-1
-1
1
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
1
1
-1
-1
-1
-1
1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1


-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
0
-1
-1
-1
-1
-1
-1
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0
0
-1
-1
0
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
0/1
0/1
0/1
0/1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
2
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
3
3
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
-1
